### Imports & Setup

In [ ]:
from Bio import Entrez, SeqIO
Entrez.email = input("Enter your email address: ")

In [ ]:
def find_in_entrez_db(db, query, n_results = 10):
    handle = Entrez.esearch(db=db, term=query, retmax=n_results)
    results = Entrez.read(handle)
    return results["IdList"]


def fetch_entrez_summary(db, ids):
    with Entrez.esummary(db=db, id=",".join(ids)) as handle:
        results = Entrez.read(handle)
        return results


def parse_entrez_summary(results, ids_map, keys_to_extract):
    fetched_genes_data = results["DocumentSummarySet"]["DocumentSummary"]
    parsed_genes_data = {}
    for entry, _id in zip(fetched_genes_data, ids_map):
       parsed_dict = {k: entry[k] for k in keys_to_extract}
       parsed_genes_data[_id] = parsed_dict
    return parsed_genes_data


def extract_result_info(records, keys):
    result = []
    for record in records:
        result.append({k: record[k] for k in keys})
    return result

In [ ]:
def run_and_parse_query_gene_db(query, n_results = 10):
    result_ids = find_in_entrez_db("gene", query, n_results=n_results)
    summary_raw = fetch_entrez_summary("gene", result_ids)
    summary_parsed = parse_entrez_summary(summary_raw, result_ids, keys_to_extract=["Name", "Chromosome", "MapLocation", "Description"])
    return summary_parsed

results = run_and_parse_query_gene_db("brca1[gene]", n_results=10)
for gene_data in list(results.items())[:10]:
    print(gene_data)


In [ ]:
results = run_and_parse_query_gene_db("BRCA1[gene] AND Homo Sapiens[orgn]", n_results=100)
for gene_data in list(results.items())[:10]:
    print(gene_data)

In [ ]:
brca1_gene_id, brca1_gene_details = list(results.items())[0]

In [ ]:
def find_link_between_dbs(dbfrom, dbto, gene_id, **kwargs):
    with Entrez.elink(dbfrom=dbfrom, db=dbto, id=gene_id, **kwargs) as handle:
        results = Entrez.read(handle)
        records_list = results[0]["LinkSetDb"][0]["Link"]
        return [record["Id"] for record in records_list]

In [ ]:
related_ids = find_link_between_dbs("gene", "omim", brca1_gene_id)
omim_summary = fetch_entrez_summary("omim", related_ids)
for record in omim_summary:
    print(record["Title"])

In [ ]:
related_ids = find_link_between_dbs("gene", "protein", brca1_gene_id)
protein_summary = fetch_entrez_summary("protein", related_ids)
for record in protein_summary:
    print(record["Gi"])

In [ ]:
protein_id = 121949022

with Entrez.efetch(db="protein", id=protein_id, rettype="gb", retmode="text") as handle:
    record = SeqIO.read(handle, "genbank")
    print(record.name)
    print(record.description)
    print(record.seq)

In [ ]:
related_ids = find_link_between_dbs("gene", "snp", brca1_gene_id, term="homo sapiens")
snp_summary = fetch_entrez_summary("snp", related_ids[:50])
summary_parsed = parse_entrez_summary(snp_summary, result_ids, keys_to_extract=["SNP_ID", "SNP_CLASS", "GENES", "CHRPOS"])
for record in summary_parsed.values():
    print(f"RecordID: {record['SNP_ID']} | Class: {record['SNP_CLASS']} | Gene: {record['GENES']} | Position: {record['CHRPOS']}")


### Wnioski

Udało mi się uzyskać następujące informacje na temat genu BRCA1:
- Gen ten występuje nie tylko u ludzi. W pierwszym query otrzymałem wyniki wyszuiwania dla wielu innych organizmów. Zawężenie zapytania do gatunku Homo Sapiens pomogło mi zlokalizować gen, który mnie interesuje.
- Na podstawie wyników wyszukiwania w bazie omim można stwierdzić, że gen produkuje białko odpowiedzialne za naprawę DNA która hamuje rozwój nowotworów piersi.
- Sądząc po wynikach wyszukiwania powiązanych sekwencji aminokwasowych gen ten koduje kilkadziesiąt takich sekwencji, które budują to białko.
- Q3LRJ6_HUMAN jest białkiem którego obecność wskazuje na możliwość występowania raka piersi u badanej osoby.
- Baza danych snp zawiera mutacje genów. Wyniki wyszukiwania wskazują, że znane ludziom jest wiele mutacji tego genu o różnych typach i lokalizacjach.